import numpy as np
# 1
a = np.array([2, 4, 6]); print(a / 2, a // 4)
# 2
m = np.ones((3, 4)); print(m.sum(axis=1))
# 3
print(np.arange(3) + np.arange(3).reshape(3, 1))
# 4
x = np.array([1, 4, 9]); print(np.sqrt(x).sum())
# 5
print((np.arange(1, 5) ** 2).mean())

Ваши ответы и объяснения:
1. [1. 2. 3.] и [0 1 1] — / делит обычно, // делит нацело.
2. [4. 4. 4.] — axis=1 суммирует четыре числа в каждой строке.
3. Матрица [[0,1,2],[1,2,3],[2,3,4]] — broadcasting складывает столбец и строку.
4. 6.0 — корни 1, 2, 3 и их сумма 6.
5. 7.5 — квадраты 1, 4, 9, 16, среднее 30/4.

In [19]:
# Проверка предсказаний
import numpy as np
a=np.array([2,4,6]);print("1:",a/2,a//4)
m=np.ones((3,4));print("2:",m.sum(axis=1))
print("3:\n",np.arange(3)+np.arange(3).reshape(3,1))
x=np.array([1,4,9]);print("4:",np.sqrt(x).sum())
print("5:",(np.arange(1,5)**2).mean())

1: [1. 2. 3.] [0 1 1]
2: [4. 4. 4.]
3:
 [[0 1 2]
 [1 2 3]
 [2 3 4]]
4: 6.0
5: 7.5


In [20]:
import csv

with open("orders.csv", encoding="utf-8-sig") as f:
    rows = list(csv.DictReader(f))

In [21]:
# Чистые данные: без дублей, без TEST, только оплаченные
_, first = np.unique(oid, return_index=True)
keep = np.zeros(len(oid), dtype=bool)
keep[first] = True
clean = keep & (cust != "TEST") & paid

print("строк всего:", len(oid), "| в чистых данных:", clean.sum())

строк всего: 5463 | в чистых данных: 4739


In [22]:
import csv
import numpy as np

CITY = {"алматы": "Алматы", "almaty": "Алматы", "астана": "Астана", "astana": "Астана",
        "шымкент": "Шымкент", "shymkent": "Шымкент", "караганда": "Караганда",
        "актобе": "Актобе", "атырау": "Атырау"}

with open("orders.csv", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

amount = np.array([float(r["amount"].replace(" ", "")) for r in rows])
paid   = np.array([r["status"] == "оплачен" for r in rows])
days   = np.array([float(r["delivery_days"]) if r["delivery_days"] else np.nan for r in rows])
city   = np.array([CITY.get(r["city"].strip().lower(), r["city"].strip()) for r in rows])
cust   = np.array([r["customer_id"] for r in rows])
month  = np.array([r["order_date"][:7] for r in rows])      # "2026-09" или "2026-10"
cat    = np.array([r["category"] for r in rows])
oid    = np.array([r["order_id"] for r in rows])

print(amount.shape, np.unique(month))

(5463,) ['2026-09' '2026-10']


In [23]:
#База
#1
sep=clean & (month=="2026-09")
oct=clean & (month=="2026-10")
sep_rev=amount[sep].sum()
oct_rev=amount[oct].sum()
change=(oct_rev-sep_rev)/sep_rev*100
print("Clean rows:",clean.sum())
print("September:",sep_rev)
print("October:",oct_rev)
print("Change:",round(change,1),"%")
print("Manager: after cleaning, revenue fell by",round(-change,1),"%")

Clean rows: 4739
September: 35243410.0
October: 34419520.0
Change: -2.3 %
Manager: after cleaning, revenue fell by 2.3 %


In [24]:
#2
categories=np.unique(cat)
months=np.unique(month)
category_table=np.array([[amount[clean & (cat==c) & (month==m)].sum() for m in months] for c in categories])
print("Categories:",categories)
print("Months:",months)
print(category_table)
total_category=category_table.sum(axis=1)
changes_category=(category_table[:,1]-category_table[:,0])/category_table[:,0]*100
print("Top category:",categories[np.argmax(total_category)],total_category.max())
print("Biggest decrease:",categories[np.argmin(changes_category)],round(changes_category.min(),1),"%")
print("Manager: the category with the biggest decrease needs attention.")

Categories: ['Дом' 'Книги' 'Красота' 'Одежда' 'Электроника']
Months: ['2026-09' '2026-10']
[[ 4997730.  5125790.]
 [ 1065300.  1140910.]
 [ 2610950.  2678990.]
 [ 9167100.  8505730.]
 [17402330. 16968100.]]
Top category: Электроника 34370430.0
Biggest decrease: Одежда -7.2 %
Manager: the category with the biggest decrease needs attention.


In [25]:
#3
count_table=np.array([[(clean & (cat==c) & (month==m)).sum() for m in months] for c in categories])
avg_table=category_table/count_table
print("Order counts:")
print(count_table)
print("Average checks:")
print(np.round(avg_table,2))
print("Top October average:",categories[np.argmax(avg_table[:,1])],round(avg_table[:,1].max(),2))
print("Manager: this category has the highest average paid order in October.")

Order counts:
[[476 476]
 [265 299]
 [422 435]
 [678 652]
 [531 505]]
Average checks:
[[10499.43 10768.47]
 [ 4020.    3815.75]
 [ 6187.09  6158.6 ]
 [13520.8  13045.6 ]
 [32772.75 33600.2 ]]
Top October average: Электроника 33600.2
Manager: this category has the highest average paid order in October.


In [26]:
#Стандарт
#4
cities=np.unique(city)
table=np.array([[amount[clean & (city==c) & (month==m)].sum() for m in months] for c in cities])
share=table/table.sum(axis=0)
print("Cities:",cities)
print("City shares:")
print(np.round(share*100,2))
print("Column sums:",share.sum(axis=0))
shymkent=share[cities=="Шымкент"][0]
print("Shymkent:",round(shymkent[0]*100,1),"% ->",round(shymkent[1]*100,1),"%")
print("Manager: Shymkent's share of monthly revenue decreased.")

Cities: ['Актобе' 'Алматы' 'Астана' 'Атырау' 'Караганда' 'Шымкент']
City shares:
[[ 7.31  7.38]
 [41.15 42.54]
 [25.63 25.71]
 [ 2.62  2.82]
 [10.35  9.8 ]
 [12.93 11.75]]
Column sums: [1. 1.]
Shymkent: 12.9 % -> 11.7 %
Manager: Shymkent's share of monthly revenue decreased.


In [27]:
#5
city_change=table[:,1]-table[:,0]
total_change=city_change.sum()
contribution=city_change/total_change*100
for i in range(len(cities)):
    print(cities[i],city_change[i],round(contribution[i],1),"%")
print("Total change:",total_change)
print("Shymkent contribution:",round(contribution[cities=="Шымкент"][0],1),"%")
print("Manager: check Shymkent first to understand the revenue decrease.")

Актобе -36540.0 4.4 %
Алматы 141950.0 -17.2 %
Астана -183690.0 22.3 %
Атырау 45870.0 -5.6 %
Караганда -276840.0 33.6 %
Шымкент -514640.0 62.5 %
Total change: -823890.0
Shymkent contribution: 62.5 %
Manager: check Shymkent first to understand the revenue decrease.


In [28]:
#6
share = table / table.sum(axis=1)[:, None]
print(np.round(share * 100, 2))

[[50.36 49.64]
 [49.76 50.24]
 [50.51 49.49]
 [48.79 51.21]
 [51.97 48.03]
 [52.99 47.01]]


В исходном коде возникает ошибка из-за несовместимости размеров массивов при broadcasting. `table.sum(axis=1)` вычисляет сумму по каждой строке, но возвращает одномерный массив. Добавление `[:, None]` превращает его в столбец, благодаря чему NumPy может правильно разделить значения каждой строки на соответствующую сумму.

In [29]:
#6
# table: rows are cities, columns are months
# table.sum(axis=1) has one number per city; it needs a new column axis
share_city=table/table.sum(axis=1).reshape(-1,1)
print("Shares inside each city:")
print(np.round(share_city*100,2))
print("Row sums:",share_city.sum(axis=1))
print("Manager: each city's September and October shares add up to 100%.")

Shares inside each city:
[[50.36 49.64]
 [49.76 50.24]
 [50.51 49.49]
 [48.79 51.21]
 [51.97 48.03]
 [52.99 47.01]]
Row sums: [1. 1. 1. 1. 1. 1.]
Manager: each city's September and October shares add up to 100%.


In [30]:
#Челлендж
#7
x=amount[clean]
z=(x-x.mean())/x.std()
outliers=(np.abs(z)>3).sum()
percent=outliers/len(z)*100
print("Mean:",round(z.mean(),6))
print("Std:",round(z.std(),6))
print("Outliers:",outliers)
print("Share:",round(percent,2),"%")
print("Manager: a small group of orders is unusually large or small.")

Mean: 0.0
Std: 1.0
Outliers: 87
Share: 1.84 %
Manager: a small group of orders is unusually large or small.


In [31]:
#8
ok=clean & ~np.isnan(days)
X=np.column_stack((amount[ok],days[ok]))
X_scaled=(X-X.min(axis=0))/(X.max(axis=0)-X.min(axis=0))
print("Shape:",X_scaled.shape)
print("Min:",X_scaled.min(axis=0))
print("Max:",X_scaled.max(axis=0))
print("Manager: scaling puts amount and delivery time on the same range, so one feature does not dominate only because its numbers are bigger.")

Shape: (4595, 2)
Min: [0. 0.]
Max: [1. 1.]
Manager: scaling puts amount and delivery time on the same range, so one feature does not dominate only because its numbers are bigger.
